# 业务流水线专章：从接收到可交付报告

> 对应主线 `book/tracks/business.md` · 练习 `book/practice/business_pipeline.md`

本课把 `BusinessPipeline` 当成「可教的教材」，不只是 demo：

1. 合成订单数据并跑通主链  
2. 读懂 `StageResult` / `RunRecord`  
3. 制造 blocked，观察不可交付  
4. 打开报告与交付索引  
5. 练习脱敏字段  

可离线运行，无需竞赛数据。


In [ ]:
from pathlib import Path
import pandas as pd
from wanglaoshi import BusinessPipeline, StageResult, intake_file

OUT = Path('outputs/business_course')
OUT.mkdir(parents=True, exist_ok=True)
SRC = OUT / 'orders.csv'

pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'order_date': pd.date_range('2024-01-01', periods=8, freq='D').astype(str),
    'amount': [10.5, 20.0, 7.2, 80.0, 15.0, 12.0, 9.0, 22.0],
    'channel': ['web', 'app', 'web', 'app', 'web', 'app', 'web', 'offline'],
    'user_tier': ['A', 'B', 'A', 'A', 'B', 'B', 'A', 'B'],
    'phone': ['13800000001', '13800000002', '13800000003', '13800000004',
              '13800000005', '13800000006', '13800000007', '13800000008'],
}).to_csv(SRC, index=False)
print('wrote', SRC.resolve())


## 1. 只跑接收（D01）

看 sha256、台账与 `period`。缺列会 `warning`，缺文件会 `blocked`。


In [ ]:
intake = intake_file(
    SRC,
    OUT / 'archive_intake',
    required_columns=['order_id', 'amount', 'channel'],
    period='2024-Q1',
)
print('status =', intake.status)
print('sha256 =', intake.data.get('sha256'))
print('period =', intake.data.get('period'))
print('warnings =', intake.warnings)


## 2. 完整主链

显式传入字段角色，避免自动猜列。关注最后一阶段是否 `deliverable=True`。


In [ ]:
pipe = BusinessPipeline(
    OUT / 'archive_run',
    required_columns=['order_id', 'amount'],
    date_col='order_date',
    value_col='amount',
    dim_cols=['channel'],
    segment_cols=['user_tier'],
    redact_fields=['phone'],
    period='2024-Q1',
    chart_unit='CNY',
    random_seed=42,
)
record = pipe.run(SRC, run_id='course01')
summary = record.summary()
print('blocked =', summary['blocked'])
for s in summary['stages']:
    print(f"{s['stage_id']:22s} {s['status']}")


## 3. 打开交付物

阅读 `report.md` 与 `delivery_index.json`。结论应能指回 evidence。


In [ ]:
import json
delivery = OUT / 'archive_run' / 'course01__delivery'
report = delivery / 'report.md'
index = delivery / 'delivery_index.json'
print(report.exists(), index.exists())
print('--- report head ---')
print(report.read_text(encoding='utf-8')[:900])
print('--- delivery_index ---')
print(json.dumps(json.loads(index.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:800])


## 4. 阻断实验

对不存在的文件跑流水线：应出现 `blocked`，且不能当作正式交付。


In [ ]:
bad = BusinessPipeline(OUT / 'archive_blocked', stop_on_block=True).run(
    OUT / 'missing.csv', run_id='blocked01'
)
print('blocked =', bad.blocked())
print([ (s.stage_id, s.status) for s in bad.stages[:4] ])
d10 = bad.stages[-1] if bad.stages else None
if d10 and d10.stage_id.endswith('archive'):
    print('deliverable =', d10.data.get('delivery_index', {}).get('deliverable'))
else:
    # stop_on_block 可能在 D01 后直接返回，尚未到 D10
    print('stages ended at', bad.stages[-1].stage_id if bad.stages else None)


## 5. 检查清单（课后）

- [ ] 能口述四种 status 的含义  
- [ ] 知道缺文件与缺列的差异  
- [ ] 打开过 report 与 delivery_index  
- [ ] 完成练习页：`book/practice/business_pipeline.md`
